# KUMPAS — Phase 2: CNN-LSTM Training (Colab)

Trains the 50-class FSL classifier on MediaPipe Holistic landmark sequences produced by the Phase 1 pipeline (`training/preprocessing/`).

**Before running:** upload **`kumpas_sequences_lite.zip`** (177MB — recommended) or `kumpas_sequences.zip` (721MB, includes pre-built augmented arrays) to Google Drive at `MyDrive/kumpas/`. With the lite zip, augmentation regenerates here from the same seed — identical arrays, identical experiment.

**Rules (PRD §7, Model/Training Agent):** every run is appended to `experiments_log.json` on Drive — architecture, hyperparams, results. No silent overwrites.

**Targets:** ≥90% held-out test accuracy (Phase 3 gate) → quantized TFLite (Phase 4).

Runtime → Change runtime type → **GPU**.

In [ ]:
# 1. Setup
import json, time, zipfile, pathlib
import numpy as np
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/kumpas')
DATA = pathlib.Path('/content/kumpas_sequences')
if not DATA.exists():
    zips = [p for p in (DRIVE / 'kumpas_sequences.zip', DRIVE / 'kumpas_sequences_lite.zip') if p.exists()]
    assert zips, f'upload kumpas_sequences.zip or kumpas_sequences_lite.zip to {DRIVE}'
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATA)
print(sorted(p.name for p in DATA.rglob('*') if p.is_file()))

In [ ]:
# 2. Load data
SEQ = next(DATA.rglob('label_map.json')).parent  # handles zip made with or without parent dir

USE_AUGMENTED = True   # X_train_aug (originals + 3x geometric augments) vs raw X_train
DROP_FACE = False      # True -> 258 features (pose+hands); latency fallback for Phase 4

if USE_AUGMENTED and not (SEQ / 'X_train_aug.npy').exists():
    # lite zip: regenerate augmentation here — same seed => same arrays as the local run
    !python {SEQ}/augment_landmarks.py --data-root {SEQ.parent} --sequences-subdir {SEQ.name} --factor 3 --seed 20260705

X_train = np.load(SEQ / ('X_train_aug.npy' if USE_AUGMENTED else 'X_train.npy'))
y_train = np.load(SEQ / ('y_train_aug.npy' if USE_AUGMENTED else 'y_train.npy'))
X_test = np.load(SEQ / 'X_test.npy')
y_test = np.load(SEQ / 'y_test.npy')
label_map = {int(k): v for k, v in json.loads((SEQ / 'label_map.json').read_text()).items()}
class_names = [label_map[i]['label'] for i in range(len(label_map))]

POSE, FACE = 33 * 4, 468 * 3
if DROP_FACE and X_train.shape[2] == 1662:
    keep = np.r_[0:POSE, POSE + FACE:1662]
    X_train, X_test = X_train[:, :, keep], X_test[:, :, keep]

N_CLASSES = len(class_names)
T, F = X_train.shape[1], X_train.shape[2]
print(f'train {X_train.shape}  test {X_test.shape}  classes {N_CLASSES}')
assert N_CLASSES == 50 and set(y_train) == set(range(50))

In [ ]:
# 3. CNN-LSTM model
from tensorflow.keras import layers, models, callbacks

def build_cnn_lstm(t, f, n_classes, conv_filters=(64, 128), lstm_units=(128, 64),
                   dense=128, dropout=0.4, lr=1e-3):
    """Conv1D blocks learn per-timestep spatial patterns across the landmark
    vector; LSTMs model the temporal dynamics of the sign. This is the
    CNN-LSTM hybrid specified in the methodology."""
    m = models.Sequential(name=f'cnnlstm_c{"-".join(map(str,conv_filters))}_l{"-".join(map(str,lstm_units))}')
    m.add(layers.Input((t, f)))
    for i, filt in enumerate(conv_filters):
        m.add(layers.Conv1D(filt, 3, padding='same', activation='relu'))
        m.add(layers.BatchNormalization())
        if i == len(conv_filters) - 1:
            m.add(layers.MaxPooling1D(2))
    for i, units in enumerate(lstm_units):
        m.add(layers.LSTM(units, return_sequences=(i < len(lstm_units) - 1)))
        m.add(layers.Dropout(dropout))
    m.add(layers.Dense(dense, activation='relu'))
    m.add(layers.Dropout(dropout))
    m.add(layers.Dense(n_classes, activation='softmax'))
    m.compile(tf.keras.optimizers.Adam(lr), 'sparse_categorical_crossentropy', ['accuracy'])
    return m

model = build_cnn_lstm(T, F, N_CLASSES)
model.summary()

In [ ]:
# 4. Train (validation split held out of TRAIN only; test set untouched until eval)
RUN_NOTES = 'baseline'  # describe what changed vs previous experiment
EPOCHS, BATCH = 120, 32

cbs = [
    callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_accuracy'),
    callbacks.ReduceLROnPlateau(patience=6, factor=0.5, monitor='val_loss'),
]
t0 = time.time()
hist = model.fit(X_train, y_train, validation_split=0.15, epochs=EPOCHS,
                 batch_size=BATCH, callbacks=cbs, verbose=2)
train_secs = time.time() - t0
val_acc = float(max(hist.history['val_accuracy']))
print(f'best val_accuracy {val_acc:.4f} in {train_secs:.0f}s')

In [ ]:
# 5. Evaluate on held-out test set (Phase 3 numbers)
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

y_prob = model.predict(X_test, verbose=0)
y_pred = y_prob.argmax(1)
test_acc = float((y_pred == y_test).mean())
print(f'TEST ACCURACY: {test_acc:.4f}  (gate: >= 0.90)')
print(classification_report(y_test, y_pred, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES), class_names, rotation=90, fontsize=7)
ax.set_yticks(range(N_CLASSES), class_names, fontsize=7)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if cm[i, j] and i != j:
            ax.text(j, i, cm[i, j], ha='center', va='center', color='red', fontsize=6)
plt.tight_layout(); plt.savefig('/content/confusion_matrix.png', dpi=150); plt.show()

# most-confused pairs -> per-gesture error analysis for the thesis
pairs = [(cm[i, j], class_names[i], class_names[j])
         for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j and cm[i, j]]
for n, a, b in sorted(pairs, reverse=True)[:15]:
    print(f'{n}x  true={a:20s} pred={b}')

In [ ]:
# 6. Log experiment (append-only; PRD rule: no silent overwriting)
log_path = DRIVE / 'experiments_log.json'
log = json.loads(log_path.read_text()) if log_path.exists() else []
log.append({
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'run_notes': RUN_NOTES,
    'model_name': model.name,
    'params': int(model.count_params()),
    'features': int(F), 'seq_len': int(T),
    'augmented': USE_AUGMENTED, 'drop_face': DROP_FACE,
    'epochs_run': len(hist.history['loss']), 'batch': BATCH,
    'best_val_accuracy': round(val_acc, 4),
    'test_accuracy': round(test_acc, 4),
    'train_seconds': round(train_secs),
})
log_path.write_text(json.dumps(log, indent=1))
print(f'{len(log)} experiments logged -> {log_path}')

In [ ]:
# 7. TFLite export + quantization (Phase 4 input)
# Run only for the SELECTED model (adviser reviews confusion matrix first — Phase 2 gate).
def to_tflite(model, quant, rep_data=None):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    # Keras LSTM usually lowers to fused TFLite ops; SELECT_TF_OPS is the fallback
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                      tf.lite.OpsSet.SELECT_TF_OPS]
    if quant == 'dynamic':
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    elif quant == 'float16':
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    return conv.convert()

out_dir = DRIVE / 'tflite'
out_dir.mkdir(exist_ok=True)
for quant in ('dynamic', 'float16'):
    blob = to_tflite(model, quant)
    p = out_dir / f'kumpas_50sign_{quant}.tflite'
    p.write_bytes(blob)
    print(p.name, f'{len(blob)/1e6:.2f} MB')
(out_dir / 'label_map.json').write_text(json.dumps({str(k): v for k, v in label_map.items()}, indent=1))
print('copy chosen .tflite + label_map.json into repo training/tflite_export/ for Phase 4 device benchmarking')

In [ ]:
# 8. Colab-side latency sanity check (INFORMATIONAL ONLY —
# the Phase 4 gate number must come from a real mid-range Android device)
interp = tf.lite.Interpreter(model_content=to_tflite(model, 'dynamic'))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
sample = X_test[:1].astype(np.float32)
interp.set_tensor(inp['index'], sample)
interp.invoke()  # warmup
times = []
for _ in range(50):
    t0 = time.perf_counter()
    interp.set_tensor(inp['index'], sample)
    interp.invoke()
    times.append((time.perf_counter() - t0) * 1000)
print(f'Colab CPU latency: median {np.median(times):.1f} ms  p95 {np.percentile(times, 95):.1f} ms')